# Frente 3 — SEMI MS4 y evaluación de métricas

**Proyecto Integrador — Avance 3**

Se implementa el método de referencia SEMI MS4 para estimar el módulo
de Young \(E\) a partir de frecuencias de resonancia de microvigas MEMS.

Se evalúan las 220 corridas almacenadas y se analiza el error relativo,
el sesgo y la dispersión bajo diferentes escenarios experimentales.


## 1. Metodología

El método utiliza la relación:

\[
E = 38.330 \, \rho \, \frac{f_1^2 L^4}{t^2},
\qquad f_1 = \frac{\omega_1}{2\pi}
\]

donde \(\rho\) es la densidad, \(L\) la longitud, \(t\) el espesor
y \(f_1\) la primera frecuencia natural.

Para \(k=3\), se combinan las estimaciones de las microvigas
cantilever consideradas por el procedimiento.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pinn_mems.metricas import cargar, tabla, resumen
from pinn_mems.nist.semi_ms4 import CONFIG, hash_metodo

HASH = hash_metodo(CONFIG)
resultados = cargar("semi_ms4", HASH)

print("Hash:", HASH)
print("Corridas encontradas:", len(resultados))
assert len(resultados) == 220, "Se esperaban 220 resultados."


Hash: db80caa786a2
Corridas encontradas: 220


## 2. Tabla de resultados

In [2]:
df = tabla(resultados)
print("Dimensiones:", df.shape)
display(df.head(10))
display(pd.DataFrame({"columna": df.columns}))


Dimensiones: (220, 15)


,caso,N,k,semilla,n_modos,sin_ruido,metodo,hash_config,convergio,razon_falla,tiempo_s,n_resoluciones,est_E,err_E,fallo
0,m0,15,1,0,1,False,semi_ms4,db80caa786a2,True,None,0.112460,None,7.000282e+10,0.000040,False
1,m0,15,1,0,1,True,semi_ms4,db80caa786a2,True,None,0.123805,None,7.001601e+10,0.000229,False
2,m0,15,1,0,3,False,semi_ms4,db80caa786a2,True,None,0.093652,None,7.000282e+10,0.000040,False
3,m0,15,1,0,3,True,semi_ms4,db80caa786a2,True,None,0.118672,None,7.001601e+10,0.000229,False
4,m0,15,1,1,1,False,semi_ms4,db80caa786a2,True,None,0.128662,None,7.006784e+10,0.000969,False
5,m0,15,1,1,1,True,semi_ms4,db80caa786a2,True,None,0.129323,None,7.001601e+10,0.000229,False
6,m0,15,1,1,3,False,semi_ms4,db80caa786a2,True,None,0.123756,None,7.006784e+10,0.000969,False
7,m0,15,1,1,3,True,semi_ms4,db80caa786a2,True,None,0.113885,None,7.001601e+10,0.000229,False
8,m0,15,1,2,1,False,semi_ms4,db80caa786a2,True,None,0.159064,None,7.002788e+10,0.000398,False
9,m0,15,1,2,1,True,semi_ms4,db80caa786a2,True,None,0.148693,None,7.001601e+10,0.000229,False


,columna
0,caso
1,N
2,k
3,semilla
4,n_modos
5,sin_ruido
6,metodo
7,hash_config
8,convergio
9,razon_falla


## 3. Resumen estadístico

In [3]:
estadisticas = resumen(df)
display(estadisticas)


corridas  sesgo_E_pct  dispersion_E_pct  tasa_exito  \
metodo   caso  k n_modos                                                        
semi_ms4 m0    1 1              40     0.022871          0.062995         1.0   
                 3              20     0.022871          0.006846         1.0   
               3 1              20     0.022871          0.004433         1.0   
                 3              20     0.022871          0.004433         1.0   
         m1_s1 1 1              10    -0.974128          0.078255         1.0   
                 3              10    -0.974128          0.078255         1.0   
               3 1              10    -1.218945          0.029657         1.0   
                 3              10    -1.218945          0.029657         1.0   
         m1_s3 1 1              10    -4.975173          0.075093         1.0   
                 3              10    -4.975173          0.075093         1.0   
               3 1              10    -6.134867          0.027817         1.0   
                 3              10    -6.134867          0.027817         1.0   
         m1_s5 1 1              10   -14.977782          0.067189         1.0   
                 3              10   -14.977782          0.067189         1.0   
               3 1              10   -17.957272          0.023539         1.0   
                 3              10   -17.957272          0.023539         1.0   

                          tiempo_mediano_s  
metodo   caso  k n_modos                    
semi_ms4 m0    1 1                0.148234  
                 3                0.112016  
               3 1                0.342433  
                 3                0.308528  
         m1_s1 1 1                0.123635  
                 3                0.114098  
               3 1                0.376157  
                 3                0.329669  
         m1_s3 1 1                0.114960  
                 3                0.110352  
               3 1                0.331303  
                 3                0.310524  
         m1_s5 1 1                0.102435  
                 3                0.104664  
               3 1                0.409553  
                 3                0.564588

## 4. Visualización del error

In [4]:
columnas_error = [
    c for c in df.columns
    if "error" in c.lower() or "sesgo" in c.lower()
]
print("Columnas de error disponibles:", columnas_error)

if columnas_error:
    columna = columnas_error[0]
    valores = pd.to_numeric(df[columna], errors="coerce").dropna()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(valores, bins=25, edgecolor="black")
    ax.set_title(f"Distribución de {columna} — SEMI MS4")
    ax.set_xlabel(columna)
    ax.set_ylabel("Frecuencia")
    ax.grid(alpha=0.25)
    plt.show()
else:
    print("Revisar las columnas para seleccionar la métrica de error.")


Columnas de error disponibles: []
Revisar las columnas para seleccionar la métrica de error.


## 5. Comparación entre escenarios

In [5]:
columnas = [
    c for c in ["metodo", "caso", "k", "n_modos"]
    if c in df.columns
]
print("Variables de agrupación:", columnas)
display(estadisticas.head(30))


Variables de agrupación: ['metodo', 'caso', 'k', 'n_modos']


corridas  sesgo_E_pct  dispersion_E_pct  tasa_exito  \
metodo   caso  k n_modos                                                        
semi_ms4 m0    1 1              40     0.022871          0.062995         1.0   
                 3              20     0.022871          0.006846         1.0   
               3 1              20     0.022871          0.004433         1.0   
                 3              20     0.022871          0.004433         1.0   
         m1_s1 1 1              10    -0.974128          0.078255         1.0   
                 3              10    -0.974128          0.078255         1.0   
               3 1              10    -1.218945          0.029657         1.0   
                 3              10    -1.218945          0.029657         1.0   
         m1_s3 1 1              10    -4.975173          0.075093         1.0   
                 3              10    -4.975173          0.075093         1.0   
               3 1              10    -6.134867          0.027817         1.0   
                 3              10    -6.134867          0.027817         1.0   
         m1_s5 1 1              10   -14.977782          0.067189         1.0   
                 3              10   -14.977782          0.067189         1.0   
               3 1              10   -17.957272          0.023539         1.0   
                 3              10   -17.957272          0.023539         1.0   

                          tiempo_mediano_s  
metodo   caso  k n_modos                    
semi_ms4 m0    1 1                0.148234  
                 3                0.112016  
               3 1                0.342433  
                 3                0.308528  
         m1_s1 1 1                0.123635  
                 3                0.114098  
               3 1                0.376157  
                 3                0.329669  
         m1_s3 1 1                0.114960  
                 3                0.110352  
               3 1                0.331303  
                 3                0.310524  
         m1_s5 1 1                0.102435  
                 3                0.104664  
               3 1                0.409553  
                 3                0.564588

## 6. Interpretación y conclusiones

- La ejecución sin ruido sirve como control de consistencia numérica.
- La estimación de SEMI MS4 depende de la frecuencia fundamental y
  de los parámetros geométricos utilizados.
- Los escenarios con perturbaciones sistemáticas permiten estudiar
  el sesgo del estimador.
- La comparación entre configuraciones permite identificar cambios
  en precisión y dispersión.

**Limitación:** SEMI MS4 es un método de referencia; no debe interpretarse
como una PINN ni como un procedimiento de identificación que utilice
necesariamente todos los modos disponibles.

Las conclusiones cuantitativas definitivas deben obtenerse de las
tablas generadas por este notebook.
